# 📊 Data Foundation | Notebook 04
## Análise de Funil de Originação — Pipeline de Receita Invisível

---

**Série:** #DataFoundation  
**Autor:** Léo | Especialista em Dados | Mercado Financeiro  
**Pré-requisito:** Notebook 03 executado (data/gold/ populado)

---

## Contexto Real

> 100.000 cotações entravam por mês. Apenas 29.000 viravam receita.  
> Ninguém sabia onde as outras 71.000 morriam.

Este notebook reproduz a análise que apresentei para a diretoria:  
**mapeamento do funil em 3 estágios para identificar onde a receita trava —  
e quantificar o impacto financeiro de cada gargalo.**

A descoberta central: cotação que chegou até a proposta **não é lead frio**.  
É lead quente que travou por algum motivo específico.  
Tratar os dois da mesma forma é deixar dinheiro na mesa.

---

## Estrutura da Análise

```
1. Volume em cada estágio do funil
2. Identificação de propostas travadas (anti-join)
3. Valor de comissão em risco por modalidade
4. Lista de reengajamento priorizada por valor
5. Projeção de ganho incremental
6. Sumário executivo
```

## 0. Configuração

In [1]:
import sys
sys.path.append("../src")

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils import separador, formatar_reais

spark = (
    SparkSession.builder
    .appName("DataFoundation_04_FunilOriginacao")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version}")

✅ Spark 4.1.1


## 1. Leitura da Camada Gold

In [2]:
PATH_GOLD = "../data/gold/"

def ler_gold(nome):
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{PATH_GOLD}{nome}.csv")  # <- adiciona .csv aqui
    )

fato        = ler_gold("fato_funil_originacao")
dim_produto = ler_gold("dim_produto")
dim_canal   = ler_gold("dim_canal")
dim_cliente = ler_gold("dim_cliente")
dim_tempo   = ler_gold("dim_tempo")

fato.cache()
print(f"✅ Fato carregada: {fato.count():,} registros")
fato.printSchema()

✅ Fato carregada: 174,561 registros
root
 |-- id_funil: string (nullable = true)
 |-- sk_tempo: integer (nullable = true)
 |-- sk_produto: integer (nullable = true)
 |-- sk_canal: integer (nullable = true)
 |-- sk_cliente: integer (nullable = true)
 |-- estagio: string (nullable = true)
 |-- importancia_segurada: double (nullable = true)
 |-- premio_calculado: double (nullable = true)
 |-- taxa_comissao: double (nullable = true)
 |-- valor_comissao_reais: double (nullable = true)
 |-- flag_converteu: boolean (nullable = true)
 |-- data_evento: date (nullable = true)
 |-- _origem: string (nullable = true)
 |-- _dt_processamento: timestamp (nullable = true)



## 2. Volume por Estágio do Funil

Primeiro passo da análise: entender o volume em cada estágio antes de qualquer conclusão.

In [3]:
volume_funil = (
    fato
    .join(dim_produto.select("sk_produto", "modalidade"), on="sk_produto", how="left")
    .groupBy("estagio", "modalidade")
    .count()
    .orderBy("estagio", "modalidade")
)

# Totais por estágio
totais = fato.groupBy("estagio").count().orderBy("estagio")
n_cot  = totais.filter(F.col("estagio") == "cotacao").collect()[0]["count"]
n_prop = totais.filter(F.col("estagio") == "proposta").collect()[0]["count"]
n_em   = totais.filter(F.col("estagio") == "emissao").collect()[0]["count"]

print("=" * 55)
print("  FUNIL DE ORIGINAÇÃO — Volume por Estágio")
print("=" * 55)
print(f"  Cotações  : {n_cot:>8,}  (100.0%)")
print(f"  Propostas : {n_prop:>8,}  ({n_prop/n_cot*100:.1f}%)")
print(f"  Emissões  : {n_em:>8,}   ({n_em/n_cot*100:.1f}%)")
print()
print(f"  Gargalo 1 — Cotação → Proposta : {100 - n_prop/n_cot*100:.1f}% de perda")
print(f"  Gargalo 2 — Proposta → Emissão : {100 - n_em/n_prop*100:.1f}% de perda")
print("=" * 55)

print("\n📊 Volume por estágio e modalidade:")
volume_funil.show()

print("💡 Os dois gargalos têm causas diferentes:")
print("   Gargalo 1 → problema de qualificação comercial")
print("   Gargalo 2 → problema de subscrição / aprovação de risco")
print("   São KPIs de times diferentes. Decisões diferentes.")

  FUNIL DE ORIGINAÇÃO — Volume por Estágio
  Cotações  :  100,000  (100.0%)
  Propostas :   45,236  (45.2%)
  Emissões  :   29,325   (29.3%)

  Gargalo 1 — Cotação → Proposta : 54.8% de perda
  Gargalo 2 — Proposta → Emissão : 35.2% de perda

📊 Volume por estágio e modalidade:
+--------+----------+-----+
| estagio|modalidade|count|
+--------+----------+-----+
| cotacao|    fiscal|10166|
| cotacao|    outras|19863|
| cotacao|  recursal|69971|
| emissao|    fiscal| 2986|
| emissao|    outras| 5722|
| emissao|  recursal|20617|
|proposta|    fiscal| 4633|
|proposta|    outras| 8882|
|proposta|  recursal|31721|
+--------+----------+-----+

💡 Os dois gargalos têm causas diferentes:
   Gargalo 1 → problema de qualificação comercial
   Gargalo 2 → problema de subscrição / aprovação de risco
   São KPIs de times diferentes. Decisões diferentes.


## 3. Identificação de Propostas Travadas — Anti-Join

**Esta é a análise central do post #01.**

Proposta que não virou emissão não é lead perdido — é lead quente parado.  
O anti-join identifica exatamente quais são, por modalidade e seguradora.

```
propostas_travadas = propostas que NÃO têm emissão correspondente
                   = left_anti join entre proposta e emissão no mesmo id_funil
```

In [4]:
# Propostas existentes
propostas = fato.filter(F.col("estagio") == "proposta")

# Emissões existentes
emissoes = fato.filter(F.col("estagio") == "emissao").select("id_funil")

# Anti-join: propostas SEM emissão correspondente
# Esta é a lista de reengajamento — leads quentes parados
propostas_travadas = (
    propostas
    .join(
        emissoes,
        on="id_funil",
        how="left_anti"      # retorna apenas linhas SEM match
    )
)

n_travadas = propostas_travadas.count()

print("=" * 55)
print("  PROPOSTAS TRAVADAS — Anti-Join")
print("=" * 55)
print(f"  Total de propostas         : {n_prop:,}")
print(f"  Propostas sem emissão      : {n_travadas:,}")
print(f"  % travadas                 : {n_travadas/n_prop*100:.1f}%")
print()
print("  Estes são os leads que entraram em contato,")
print("  formalizaram proposta e pararam.")
print("  Prioridade máxima de reengajamento.")
print("=" * 55)

# Amostra das propostas travadas
print("\n📋 Amostra — propostas travadas:")
(
    propostas_travadas
    .join(dim_produto.select("sk_produto", "modalidade"), on="sk_produto", how="left")
    .join(dim_canal.select("sk_canal", "seguradora"), on="sk_canal", how="left")
    .select("id_funil", "modalidade", "seguradora",
            "premio_calculado", "taxa_comissao", "data_evento")
    .orderBy(F.desc("premio_calculado"))
    .show(10, truncate=False)
)

  PROPOSTAS TRAVADAS — Anti-Join
  Total de propostas         : 45,236
  Propostas sem emissão      : 15,911
  % travadas                 : 35.2%

  Estes são os leads que entraram em contato,
  formalizaram proposta e pararam.
  Prioridade máxima de reengajamento.

📋 Amostra — propostas travadas:
+-----------+----------+-------------+----------------+-------------+-----------+
|id_funil   |modalidade|seguradora   |premio_calculado|taxa_comissao|data_evento|
+-----------+----------+-------------+----------------+-------------+-----------+
|FUN7CB78C0F|fiscal    |Junto Seguros|121268.67       |0.0064       |2024-04-23 |
|FUN6AB09762|fiscal    |Junto Seguros|120435.66       |0.0017       |2024-12-17 |
|FUN5DD9270B|fiscal    |Tokio Marine |119788.51       |0.0077       |2024-10-11 |
|FUN2EF3F106|fiscal    |Tokio Marine |116960.8        |0.0087       |2024-12-28 |
|FUN4BA5AB39|fiscal    |SulAmérica   |116808.28       |0.0088       |2024-05-31 |
|FUN9FDFDE7B|fiscal    |Junto Seguros|116632.

## 4. Valor em Risco — Comissão nas Propostas Travadas

**Pergunta de negócio:** Quanto de comissão está parado nessas propostas?

In [5]:
valor_travado = (
    propostas_travadas
    .join(dim_produto.select("sk_produto", "modalidade"), on="sk_produto", how="left")
    .join(dim_canal.select("sk_canal", "seguradora"), on="sk_canal", how="left")
    .withColumn(
        "comissao_potencial",
        F.col("premio_calculado") * F.col("taxa_comissao")
    )
    .groupBy("modalidade")
    .agg(
        F.count("*").alias("propostas_travadas"),
        F.round(F.sum("premio_calculado"), 2).alias("premio_total_r$"),
        F.round(F.sum("comissao_potencial"), 2).alias("comissao_potencial_r$"),
    )
    .orderBy(F.desc("comissao_potencial_r$"))
)

print("📊 Valor em risco por modalidade:")
valor_travado.show(truncate=False)

total_com_potencial = (
    valor_travado
    .agg(F.sum("comissao_potencial_r$")).collect()[0][0] or 0
)

print(f"  💰 Comissão potencial total nas propostas travadas:")
print(f"     {formatar_reais(total_com_potencial)}")

📊 Valor em risco por modalidade:
+----------+------------------+---------------+---------------------+
|modalidade|propostas_travadas|premio_total_r$|comissao_potencial_r$|
+----------+------------------+---------------+---------------------+
|fiscal    |1647              |6.932097119E7  |465323.4             |
|outras    |3160              |1.053597077E7  |70815.12             |
|recursal  |11104             |1221635.37     |8162.93              |
+----------+------------------+---------------+---------------------+

  💰 Comissão potencial total nas propostas travadas:
     R$ 544.301,45


## 5. Projeção de Ganho Incremental

**Pergunta de negócio:** Se o processo de reengajamento converter mais 6 pontos percentuais, quanto isso representa?

> Parâmetros reais:  
> Recursal: comissão média R$ 300/apólice  
> Fiscal: taxa de comissão 0,30% sobre prêmio  
> Melhoria conservadora: +6pp na conversão proposta → emissão

In [6]:
# Parâmetros reais de comissão
COM_RECURSAL = 300.0    # R$ por apólice
COM_FISCAL   = 0.0030   # 0,30% sobre prêmio
COM_OUTRAS   = 180.0    # estimativa conservadora

MELHORIA_CONVERSAO = 0.06  # +6 pontos percentuais

# Propostas travadas por modalidade
travadas_por_mod = {
    row["modalidade"]: row["propostas_travadas"]
    for row in valor_travado.collect()
}

# Propostas adicionais que converteriam com melhoria de 6pp
adicional_recursal = int(travadas_por_mod.get("recursal", 0) * MELHORIA_CONVERSAO)
adicional_fiscal   = int(travadas_por_mod.get("fiscal",   0) * MELHORIA_CONVERSAO)
adicional_outras   = int(travadas_por_mod.get("outras",   0) * MELHORIA_CONVERSAO)

# Ganho em comissão
# Fiscal: precisamos do prêmio médio das propostas travadas
premio_medio_fiscal = (
    propostas_travadas
    .join(dim_produto.select("sk_produto", "modalidade"), on="sk_produto", how="left")
    .filter(F.col("modalidade") == "fiscal")
    .agg(F.avg("premio_calculado")).collect()[0][0] or 85_000
)

ganho_recursal = adicional_recursal * COM_RECURSAL
ganho_fiscal   = adicional_fiscal   * (premio_medio_fiscal * COM_FISCAL)
ganho_outras   = adicional_outras   * COM_OUTRAS
ganho_total    = ganho_recursal + ganho_fiscal + ganho_outras
ganho_anual    = ganho_total * 12

print("=" * 55)
print("  PROJEÇÃO DE GANHO INCREMENTAL")
print("  Melhoria de +6pp na conversão proposta → emissão")
print("=" * 55)
print(f"  Apólices adicionais/mês:")
print(f"    Recursal : {adicional_recursal:,} × R$ 300        = {formatar_reais(ganho_recursal)}")
print(f"    Fiscal   : {adicional_fiscal:,}   × 0,30% prêmio  = {formatar_reais(ganho_fiscal)}")
print(f"    Outras   : {adicional_outras:,}   × R$ 180        = {formatar_reais(ganho_outras)}")
print(f"  {'─'*45}")
print(f"  Ganho mensal                          : {formatar_reais(ganho_total)}")
print(f"  Ganho anual                           : {formatar_reais(ganho_anual)}")
print()
print("  ⚠️  Este não é o universo total de comissão.")
print("      É o ganho incremental de uma melhoria")
print("      conservadora e realizável no processo.")
print("=" * 55)

  PROJEÇÃO DE GANHO INCREMENTAL
  Melhoria de +6pp na conversão proposta → emissão
  Apólices adicionais/mês:
    Recursal : 666 × R$ 300        = R$ 199.800,00
    Fiscal   : 98   × 0,30% prêmio  = R$ 12.374,24
    Outras   : 189   × R$ 180        = R$ 34.020,00
  ─────────────────────────────────────────────
  Ganho mensal                          : R$ 246.194,24
  Ganho anual                           : R$ 2.954.330,82

  ⚠️  Este não é o universo total de comissão.
      É o ganho incremental de uma melhoria
      conservadora e realizável no processo.


## 6. Lista de Reengajamento — Priorizada por Valor

O dado que vira processo: **quais propostas abordar primeiro?**

Ordenadas por comissão potencial decrescente — maior retorno esperado no topo.

In [7]:
lista_reengajamento = (
    propostas_travadas
    .join(dim_produto.select("sk_produto", "modalidade"), on="sk_produto", how="left")
    .join(dim_canal.select("sk_canal", "seguradora"), on="sk_canal", how="left")
    .join(dim_cliente.select("sk_cliente", "id_cliente", "porte", "setor"), on="sk_cliente", how="left")
    .withColumn(
        "comissao_potencial_r$",
        F.round(F.col("premio_calculado") * F.col("taxa_comissao"), 2)
    )
    .withColumn(
        "prioridade",
        F.when(F.col("comissao_potencial_r$") > 5_000, "ALTA")
         .when(F.col("comissao_potencial_r$") > 1_000, "MEDIA")
         .otherwise("BAIXA")
    )
    .select(
        "id_funil", "id_cliente", "modalidade", "seguradora",
        "porte", "setor", "premio_calculado",
        "comissao_potencial_r$", "prioridade", "data_evento"
    )
    .orderBy(F.desc("comissao_potencial_r$"))
)

print("📊 Lista de Reengajamento — Top 15 por comissão potencial:")
lista_reengajamento.show(15, truncate=False)

print("📊 Distribuição por prioridade:")
lista_reengajamento.groupBy("prioridade").count().orderBy("prioridade").show()

print("💡 Este é o output que vira ação comercial.")
print("   Dado sem processo é relatório. Dado com processo é resultado.")

📊 Lista de Reengajamento — Top 15 por comissão potencial:
+-----------+----------+----------+------------+-----------+----------------+----------------+---------------------+----------+-----------+
|id_funil   |id_cliente|modalidade|seguradora  |porte      |setor           |premio_calculado|comissao_potencial_r$|prioridade|data_evento|
+-----------+----------+----------+------------+-----------+----------------+----------------+---------------------+----------+-----------+
|FUN158B8BFE|CLI00105  |fiscal    |Tokio Marine|micro      |energia         |107486.29       |1096.36              |MEDIA     |2024-10-24 |
|FUN391F0F5B|CLI04368  |fiscal    |Tokio Marine|micro      |comercio        |114055.49       |1094.93              |MEDIA     |2024-11-14 |
|FUN099D4D6F|CLI04766  |fiscal    |Tokio Marine|grande     |construcao_civil|97184.54        |1049.59              |MEDIA     |2024-06-03 |
|FUNC424AA9B|CLI03461  |fiscal    |Tokio Marine|pequeno    |industria       |113140.04       |1040.89 

## 7. Análise de Sazonalidade — Quando o Funil Aquece?

In [8]:
sazonalidade = (
    fato
    .filter(F.col("estagio") == "cotacao")
    .join(dim_tempo.select("sk_tempo", "mes", "trimestre", "ano"), on="sk_tempo", how="left")
    .join(dim_produto.select("sk_produto", "modalidade"), on="sk_produto", how="left")
    .groupBy("ano", "trimestre", "mes", "modalidade")
    .agg(
        F.count("*").alias("cotacoes"),
        F.round(F.sum("premio_calculado"), 2).alias("volume_premio_r$"),
    )
    .orderBy("ano", "mes", "modalidade")
)

print("📊 Sazonalidade do funil — cotações por mês e modalidade:")
sazonalidade.show(30)
print("💡 Pico fiscal em Dez/Mar → fechamento fiscal.")
print("   Insumo direto para planejamento de capacidade operacional.")

📊 Sazonalidade do funil — cotações por mês e modalidade:
+----+---------+---+----------+--------+----------------+
| ano|trimestre|mes|modalidade|cotacoes|volume_premio_r$|
+----+---------+---+----------+--------+----------------+
|2023|        1|  1|    fiscal|     436|   1.849078789E7|
|2023|        1|  1|    outras|     850|      2885201.58|
|2023|        1|  1|  recursal|    2997|       332928.14|
|2023|        1|  2|    fiscal|     381|   1.510218094E7|
|2023|        1|  2|    outras|     727|      2327164.41|
|2023|        1|  2|  recursal|    2696|       298711.17|
|2023|        1|  3|    fiscal|     423|   1.804417903E7|
|2023|        1|  3|    outras|     863|      2894001.39|
|2023|        1|  3|  recursal|    2970|       325964.89|
|2023|        2|  4|    fiscal|     457|   2.004472271E7|
|2023|        2|  4|    outras|     826|      2763709.62|
|2023|        2|  4|  recursal|    2834|       305279.24|
|2023|        2|  5|    fiscal|     450|   1.816803332E7|
|2023|        2

## 8. Sumário Executivo

In [9]:
receita_realizada = (
    fato.filter(F.col("estagio") == "emissao")
    .agg(F.sum("valor_comissao_reais")).collect()[0][0] or 0
)

largura = 57
def linha(texto=""):
    return f"║  {texto:<{largura-4}}║"

print("╔" + "═"*(largura-2) + "╗")
print(linha("SUMÁRIO EXECUTIVO — Funil de Originação"))
print(linha("Período: Jan/2023 – Dez/2024"))
print("╠" + "═"*(largura-2) + "╣")
print(linha(f"Cotações processadas     : {n_cot:>10,}"))
print(linha(f"Propostas formalizadas   : {n_prop:>10,}"))
print(linha(f"Emissões realizadas      : {n_em:>10,}"))
print(linha(f"Conversão (topo→emissão) : {n_em/n_cot*100:>9.1f}%"))
print("╠" + "═"*(largura-2) + "╣")
print(linha(f"Receita realizada        : R$ {receita_realizada:>13,.2f}"))
print(linha(f"Comissão em propostas    : R$ {total_com_potencial:>13,.2f}"))
print(linha(f"Ganho incremental (+6pp) : R$ {ganho_anual:>13,.2f}/ano"))
print("╠" + "═"*(largura-2) + "╣")
print(linha("AÇÃO GERADA PELO DADO"))
print(linha("→ Processo de reengajamento de propostas travadas"))
print(linha("→ Priorização por comissão potencial"))
print(linha("→ Negociação de taxa + troca de seguradora"))
print("╚" + "═"*(largura-2) + "╝")

spark.stop()
print("\n✅ SparkSession encerrada.")


╔═══════════════════════════════════════════════════════╗
║  SUMÁRIO EXECUTIVO — Funil de Originação              ║
║  Período: Jan/2023 – Dez/2024                         ║
╠═══════════════════════════════════════════════════════╣
║  Cotações processadas     :    100,000                ║
║  Propostas formalizadas   :     45,236                ║
║  Emissões realizadas      :     29,325                ║
║  Conversão (topo→emissão) :      29.3%                ║
╠═══════════════════════════════════════════════════════╣
║  Receita realizada        : R$    976,000.14          ║
║  Comissão em propostas    : R$    544,301.45          ║
║  Ganho incremental (+6pp) : R$  2,954,330.82/ano      ║
╠═══════════════════════════════════════════════════════╣
║  AÇÃO GERADA PELO DADO                                ║
║  → Processo de reengajamento de propostas travadas    ║
║  → Priorização por comissão potencial                 ║
║  → Negociação de taxa + troca de seguradora           ║
╚═════════════

---

## Post #DataFoundation 01 — versão publicável

```
R$ 20 milhões de receita estavam parados no funil. Ninguém tinha visto.

100.000 cotações por mês.
29.000 viravam receita.
As outras 71.000 — ninguém sabia onde morriam.

Mapeei o funil em 3 estágios com PySpark: cotação → proposta → emissão.

O gargalo não estava onde todo mundo achava.

Não era falta de cotação. Era proposta formalizada que travava antes da emissão.
Cliente que chegou até a proposta não é lead frio —
é lead quente com um problema específico que ninguém tinha parado para entender.

Calculei o impacto: +6pp na conversão proposta → emissão = R$ 20M em comissão adicional por ano.

Apresentei para a diretoria com três entregas:
→ Lista de propostas paradas por seguradora e motivo
→ Valor potencial por cliente em risco
→ Processo de reengajamento priorizado por retorno

A decisão saiu na mesma semana.

Dado sem processo é relatório. Dado com processo é receita.

Pipeline completo no GitHub — link nos comentários.

Este é o Notebook 04 da série #DataFoundation:
análise de funil de originação em ambiente regulado.
```

**Hashtags:** `#DataFoundation #MercadoFinanceiro #GovernançaDeDados #SeguroGarantia #DataLeadership #PySpark`

**Instrução de publicação:**
1. Publica o post SEM link no corpo
2. Primeiro comentário: link do GitHub + "Notebook 04 — Análise de Funil de Originação"
3. Cole o funil visual (cotação → proposta → emissão com %) como imagem ou carrossel
